In [15]:
import os
import numpy as np
import pandas as pd
import lightkurve as lk
from lightkurve import LightCurveCollection
from tqdm import tqdm

In [16]:
def process_lightcurve(target_id, mission, period, epoch,
                       n_global=2001, n_local=201, local_window=0.05,
                       fast_mode=True, timeout=30): # 👈 New parameter
    """
    Processes a light curve for a given target with a network timeout.
    """
    try:
        search = lk.search_lightcurve(target_id, mission=mission)
        if len(search) == 0:
            print(f"[WARN] {target_id}: no light curve products")
            return None, None

        # Download
        if fast_mode:
            # Pass the timeout parameter to the download method
            lc = search.download(timeout=timeout).remove_nans().flatten() # 👈 Timeout added
        else:
            lcs = []
            for prod in search:
                try:
                    # Pass the timeout parameter here as well
                    dl = prod.download(timeout=timeout) # 👈 Timeout added
                    if dl is not None:
                        lcs.append(dl)
                except Exception:
                    pass
            if len(lcs) == 0:
                print(f"[WARN] {target_id}: all downloads failed")
                return None, None

            if len(lcs) == 1:
                lc = lcs[0].remove_nans().flatten()
            else:
                lc = LightCurveCollection(lcs).stitch().remove_nans().flatten()

        # Ephemeris check
        if period is None or np.isnan(period) or epoch is None or np.isnan(epoch):
            print(f"[WARN] {target_id} invalid period/epoch")
            return None, None

        # Fold + normalize
        folded = lc.fold(period=period, epoch_time=epoch)
        phase, flux = folded.phase.value, folded.flux.value
        flux = (flux - np.nanmedian(flux)) / np.nanstd(flux)

        order = np.argsort(phase)
        phase, flux = phase[order], flux[order]

        # Global view
        global_phase = np.linspace(-0.5, 0.5, n_global)
        global_flux  = np.interp(global_phase, phase, flux)

        # Local view
        mask = (phase > -local_window) & (phase < local_window)
        if np.sum(mask) < 3:
            print(f"[WARN] {target_id} too few points in transit")
            return None, None
        local_phase = np.linspace(-local_window, local_window, n_local)
        local_flux  = np.interp(local_phase, phase[mask], flux[mask])

        return global_flux, local_flux

    except Exception as e:
        # This block will now also catch timeout errors
        print(f"[FAIL] {target_id}: {e}")
        return None, None

In [17]:
def balance_catalog(df, catalog, n_samples=None):
    """Ensure 50/50 pos/neg for a catalog."""
    df = df[df["catalog"]==catalog]
    df = df[df["period"].notnull() & df["epoch"].notnull()]

    positives = df[df["label"]==1]
    negatives = df[df["label"]==0]

    n = min(len(positives), len(negatives))
    if n_samples:
        n = min(n, n_samples)

    pos_sample = positives.sample(n, random_state=42)
    neg_sample = negatives.sample(n, random_state=42)

    return pd.concat([pos_sample, neg_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

In [18]:
def build_batches_for_catalog(df, catalog_name,
                              batch_size=50, out_dir="batches",
                              fast_mode=True):
    """
    Build batched dataset for a single catalog (KOI or TOI) with a progress bar.
    """
    os.makedirs(out_dir, exist_ok=True)

    balanced = balance_catalog(df, catalog_name)
    batches = [balanced.iloc[i:i+batch_size]
               for i in range(0, len(balanced), batch_size)]

    batch_files = []
    mission = "Kepler" if catalog_name == "KOI" else "TESS"
    id_prefix = "KIC" if catalog_name == "KOI" else "TIC"

    print(f"[START] {catalog_name}: {len(balanced)} balanced rows → {len(batches)} batches")

    # 👇 Wrap the 'batches' iterable with tqdm to create a progress bar
    for bi, batch_df in enumerate(tqdm(batches, desc=f"Processing {catalog_name}"), 1):
        out_file = os.path.join(out_dir, f"{catalog_name}_batch_{bi:03d}.npz")
        if os.path.exists(out_file):
            # tqdm.write is used to print messages without disturbing the bar
            tqdm.write(f"[SKIP] {out_file} exists")
            batch_files.append(out_file)
            continue

        Xg, Xl, y = [], [], []
        # This print statement is no longer needed as tqdm shows the progress
        # print(f"[BATCH] {catalog_name} {bi}/{len(batches)} ({len(batch_df)} targets)")
        
        for _, row in batch_df.iterrows():
            target = f"{id_prefix} {int(row['target_id'])}"
            g, l = process_lightcurve(target, mission, row["period"], row["epoch"], fast_mode=fast_mode)
            if g is not None and l is not None:
                Xg.append(g)
                Xl.append(l)
                y.append(row["label"])

        np.savez(out_file, X_global=np.array(Xg, np.float32),
                 X_local=np.array(Xl, np.float32),
                 y=np.array(y, np.int8))
        tqdm.write(f"[SAVE] {out_file} with {len(y)} samples") # Use tqdm.write here too
        batch_files.append(out_file)

    return batch_files

In [19]:
all_df = pd.read_csv("../data/all_df.csv")

C:\Users\yashs\AppData\Local\Temp\ipykernel_1828\3349140982.py:1: DtypeWarning: Columns (2,3,6,9,26,27,28,29,30,36,41,105,131,132,134,135,136,137,141,152,159,163,169,173,241,242,243) have mixed types. Specify dtype option on import or set low_memory=False.
  all_df = pd.read_csv("../data/all_df.csv")


In [20]:
print(f"Data type of 'target_id': {all_df['target_id'].dtype}")
print(f"Data type of period column: {all_df['period'].dtype}")
print(f"Data type of epoch column: {all_df['epoch'].dtype}")

Data type of 'target_id': int64
Data type of period column: float64
Data type of epoch column: float64


In [ ]:
koi_batches = build_batches_for_catalog(all_df, "KOI",
                                        batch_size=50,
                                        out_dir="batches/KOI",
                                        fast_mode=False)

[START] KOI: 5492 balanced rows → 110 batches


Processing KOI:   0%|                                                                          | 0/110 [00:00<?, ?it/s]

[WARN] KIC 2832589: all downloads failed
